# Practice 10 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime Operations
A DataFrame of employee records has been created for you.

1. Convert `hire_date` to datetime using `pd.to_datetime()`
2. Extract the hire `year` and `month` as new columns
3. Add a `tenure_days` column = number of days since hire (use `pd.Timestamp('today')` as today's date)
4. Use `np.mean()` and `np.median()` to summarise `tenure_days`

In [56]:
# Starter data — don't change this
employees = pd.DataFrame({
    'name':      ['Alice', 'Bob', 'Carol', 'Dave', 'Emma', 'Frank', 'Grace', 'Hank'],
    'hire_date': ['2018-03-15', '2020-07-01', '2019-11-22', '2021-02-10',
                  '2017-06-30', '2023-01-05', '2022-09-18', '2016-04-25'],
    'salary':    [72000, 58000, 65000, 48000, 81000, 54000, 67000, 90000],
    'department':['Engineering', 'Marketing', 'Engineering', 'Sales',
                  'Engineering', 'Sales', 'Marketing', 'Engineering'],
})
# Your code here
employees['hire_date'] = pd.to_datetime(employees['hire_date'])
employees['year'] = employees['hire_date'].dt.year
employees['month'] = employees['hire_date'].dt.month
employees['tenure'] = (pd.Timestamp('today')-employees['hire_date']).dt.days
np.mean(employees['tenure'])
np.median(employees['tenure'])

np.float64(2187.0)

### Task 2: Handling Missing Data
A DataFrame with missing values has been created for you.

1. Use `.isna().sum()` to count missing values per column
2. Use `np.where()` to add a `has_missing` column — `True` if a row has any missing value, `False` otherwise (hint: `.isna().any(axis=1)`)
3. Fill missing `score` values with the column median using `.fillna()`
4. Fill missing `grade` values with `'Unknown'`
5. Use `.dropna(subset=['city'])` to drop rows where `city` is missing

In [14]:
# Starter data — don't change this
records = pd.DataFrame({
    'name':  ['Ana', 'Ben', 'Cara', 'Dan', 'Eve', 'Finn', 'Gia', 'Hal'],
    'score': [88.0, np.nan, 74.0, np.nan, 91.0, 65.0, np.nan, 79.0],
    'grade': ['B', 'A', np.nan, 'C', 'A', np.nan, 'B', np.nan],
    'city':  ['London', 'Paris', np.nan, 'Berlin', 'London', 'Paris', 'Berlin', np.nan],
})

# Your code here
records.isna().sum()
records['has_missing'] = np.where(records.isna().any(axis =1),True, False)
records['score'] = records['score'].fillna(records['score'].median())
records['grade'] = records['grade'].fillna('Unknown')
records = records.dropna(subset=['city'])
records

,name,score,grade,city,has_missing
0,Ana,88.0,B,London,False
1,Ben,79.0,A,Paris,True
3,Dan,79.0,C,Berlin,True
4,Eve,91.0,A,London,False
5,Finn,65.0,Unknown,Paris,True
6,Gia,79.0,B,Berlin,True


---
## Level 2 — Transformations

### Task 3: Merging DataFrames
Two DataFrames have been created for you.

1. Use `pd.merge()` with `how='inner'` to join `orders` and `customers` on `customer_id`
2. Use `pd.merge()` with `how='left'` to keep all orders, including those with no matching customer
3. From the left join result, filter to rows where `name` is `NaN` — these are orders with no matching customer
4. Use `np.setdiff1d()` to find `customer_id` values that appear in `orders` but not in `customers`

In [59]:
# Starter data — don't change this
orders = pd.DataFrame({
    'order_id':    [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 2, 3, 1, 5, 6],
    'amount':      [250, 480, 130, 390, 210, 540],
})
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave'],
    'city':        ['London', 'Paris', 'Berlin', 'Rome'],
})

# Your code here
m=pd.merge(orders, customers, how = 'inner', on='customer_id')
lm = pd.merge(orders, customers, how='left', on='customer_id')

no_match = lm[lm['name'].isna()]
no_match

#np.setdiff1d(orders['customer_id'], customers['customer_id'])



,order_id,customer_id,amount,name,city
4,105,5,210,NaN,NaN
5,106,6,540,NaN,NaN


### Task 4: map(), qcut(), and clip()
A DataFrame of job applications has been created for you.

1. Use `.map()` with a dictionary to add a `status_code` column: `'Applied'→1`, `'Interview'→2`, `'Offer'→3`, `'Rejected'→0`
2. Use `pd.qcut()` to bin `score` into 4 equal-sized groups, label them `['Q1','Q2','Q3','Q4']`, store as `quartile`
3. Use `np.clip()` to cap `score` between 40 and 95, store as `score_clipped`

In [58]:
# Starter data — don't change this
np.random.seed(3)
applications = pd.DataFrame({
    'candidate': [f'C{i}' for i in range(1, 13)],
    'status':    np.random.choice(['Applied', 'Interview', 'Offer', 'Rejected'], size=12),
    'score':     np.random.randint(20, 100, size=12),
})

# Your code here
smap = {'Applied':1,
        'Interview':2,
        'Offer': 3,
        'Rejected': 0}

applications['status_code'] = applications['status'].map(smap)
applications['quartile'] = pd.qcut(applications['score'],4,labels=['Q1','Q2','Q3','Q4'])
applications['score_clipped'] = np.clip(applications['score'],40,95)

---
## Level 3 — Aggregation

### Task 5: pivot_table
A DataFrame of store sales has been created for you.

1. Use `pd.pivot_table()` to show mean `revenue` for each `store` × `category` combination
2. Repeat with `aggfunc=['mean', 'sum']` to get both stats at once
3. Add `margins=True` to include row and column totals
4. Use `np.round()` to round the mean-only pivot table to 1 decimal place

In [35]:
# Starter data — don't change this
np.random.seed(7)
store_sales = pd.DataFrame({
    'store':    np.random.choice(['Store A', 'Store B', 'Store C'], size=30),
    'category': np.random.choice(['Electronics', 'Clothing', 'Food'], size=30),
    'revenue':  np.random.randint(100, 2000, size=30).astype(float),
})

# Your code here
p = pd.pivot_table(store_sales, index='store',columns='category',
               values='revenue')
pd.pivot_table(store_sales, index='store',columns='category',
               values='revenue', aggfunc=['mean','sum'])
pd.pivot_table(store_sales, index='store',columns='category',
               values='revenue', margins=True)
np.round(pd.pivot_table(store_sales, index='store',columns='category',
               values='revenue'),1)
p.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, Store A to Store C
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Clothing     3 non-null      float64
 1   Electronics  3 non-null      float64
 2   Food         3 non-null      float64
dtypes: float64(3)
memory usage: 96.0+ bytes


### Task 6: resample() with Time Series
A DataFrame of daily sales over 60 days has been created for you.

1. Set `date` as the index
2. Use `.resample('W').sum()` to get weekly total `sales`
3. Use `.resample('W').mean()` to get weekly average `sales`
4. Add a `cumulative` column to the weekly totals using `.cumsum()`
5. Use `np.argmax()` to find which week had the highest total sales, then print that row

In [48]:
# Starter data — don't change this
np.random.seed(11)
daily_sales = pd.DataFrame({
    'date':  pd.date_range('2024-01-01', periods=60, freq='D'),
    'sales': np.random.randint(200, 1500, size=60),
})

# Your code here

daily_sales = daily_sales.set_index('date')
wt = daily_sales.resample('W').sum()
daily_sales.resample('W').mean()
wt['cumulative'] = wt.cumsum()
wt.iloc[np.argmax(wt['sales'])]

sales          8054
cumulative    29111
Name: 2024-01-28 00:00:00, dtype: int64

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Load the Titanic dataset and run a full analysis:

1. Drop rows where `age` or `fare` is missing
2. Add a `fare_per_age` column = `fare / age`
3. Use `pd.cut()` to bin `age` into: Child (<18), Adult (18–60), Senior (60+), store as `age_group`
4. Use `.groupby('age_group').agg()` to get mean and std of `fare` per age group
5. Use `.groupby('pclass')['survived'].transform('mean')` to add a `class_survival_rate` column
6. Use `np.corrcoef()` to find the correlation between `age` and `fare`
7. Use `pd.pivot_table()` to show survival rate (`survived` mean) for each `pclass` × `sex` combination

In [ ]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
titanic = pd.read_csv(url)

# Your code here
titanic = titanic.dropna(subset=['age','fare'])
titanic['fare_per_age'] = titanic['fare']/titanic['age']
titanic['age_group'] = pd.cut(titanic['age'],bins=[0,18,60,np.inf],
                              labels=['Child','Adult','Senoir'])
titanic.groupby('age_group',observed=True)['fare'].agg(['mean','std'])
titanic['class_survival_rate']= titanic.groupby('pclass')['survived'].transform('mean')
np.corrcoef(titanic['age'],titanic['fare'])
pd.pivot_table(titanic,index='pclass', columns='sex',values='survived')


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_55603/3207759863.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby('age_group')['fare'].agg(['mean','std'])


sex,female,male
pclass,,
1,0.964706,0.396040
2,0.918919,0.151515
3,0.460784,0.150198
